In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Tuple, Dict
import logging
import tqdm

class CanaryExposureCalculator:
    def __init__(self, model_name: str):
        """
        Initialize the exposure calculator with a pre-trained language model.
        
        Args:
            model_name: HuggingFace model identifier
        """
        logging.info(f"Loading model and tokenizer: {model_name}")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        logging.info(f"Model loaded on {self.device}")

    def generate_canaries(self, format_string: str, randomness_space: List, num_canaries: int = 100) -> List[str]:
        """
        Generate canaries based on a format string with placeholders.
        
        Args:
            format_string: String with "_" as placeholders
            randomness_space: List of possible values to fill the placeholders
            num_canaries: Number of canaries to generate
            
        Returns:
            List of generated canary strings
        """
        canaries = []
        
        # Count placeholders in format string
        num_placeholders = format_string.count("_")
        
        # Generate random canaries
        for _ in range(num_canaries):
            canary = format_string
            for _ in range(num_placeholders):
                random_value = np.random.choice(randomness_space)
                canary = canary.replace("_", str(random_value), 1)
            canaries.append(canary)
            
        return canaries
    
    def calculate_ranks(self, format_string: str, randomness_space: List, 
                        canaries: List[str]) -> Dict[str, int]:
        """
        Calculate the rank of each canary among all possible completions.
        
        Args:
            format_string: The format string with placeholders
            randomness_space: List of possible values for placeholders
            canaries: List of canary strings to rank
            
        Returns:
            Dictionary mapping each canary to its rank
        """
        # Get the prefix (part before the first placeholder)
        prefix = format_string.split("_")[0]
        
        # Tokenize the prefix
        prefix_tokens = self.tokenizer.encode(prefix, return_tensors="pt").to(self.device)
        
        # Get probabilities for all possible next tokens
        with torch.no_grad():
            outputs = self.model(prefix_tokens)
            next_token_logits = outputs.logits[:, -1, :]
            next_token_probs = torch.softmax(next_token_logits, dim=-1)
        
        # Get completions for all canaries
        completions = [canary[len(prefix):] for canary in canaries]
        
        # Calculate likelihood of each completion
        canary_scores = {}
        for i, completion in enumerate(tqdm.tqdm(completions)):
            completion_tokens = self.tokenizer.encode(completion, add_special_tokens=False)
            
            # Calculate likelihood score for this completion
            score = 0
            current_input = prefix_tokens.clone()
            
            for token in completion_tokens:
                with torch.no_grad():
                    outputs = self.model(current_input)
                    next_token_logits = outputs.logits[:, -1, :]
                    next_token_probs = torch.softmax(next_token_logits, dim=-1)
                    
                    # Get probability of this token
                    token_prob = next_token_probs[0, token].item()
                    score += np.log(token_prob) if token_prob > 0 else -1000
                    
                    # Add token to input for next iteration
                    current_input = torch.cat([current_input, 
                                             torch.tensor([[token]]).to(self.device)], dim=1)
            
            canary_scores[canaries[i]] = score
        
        # Sort canaries by score (higher is better)
        sorted_canaries = sorted(canary_scores.keys(), key=lambda k: canary_scores[k], reverse=True)
        
        # Assign ranks
        canary_ranks = {canary: sorted_canaries.index(canary) + 1 for canary in canaries}
        
        return canary_ranks
    
    def calculate_exposure(self, format_string: str, randomness_space: List, 
                          target_canaries: List[str], num_samples: int = 1000) -> Dict[str, float]:
        """
        Calculate the exposure metric for target canaries.
        
        Args:
            format_string: The format string with placeholders
            randomness_space: List of possible values for placeholders
            target_canaries: List of canaries to calculate exposure for
            num_samples: Number of random canaries to sample for ranking
            
        Returns:
            Dictionary mapping each target canary to its exposure value
        """
        # Generate additional random canaries to establish rankings
        random_canaries = self.generate_canaries(format_string, randomness_space, num_samples)
        
        # Combine target and random canaries
        all_canaries = target_canaries + random_canaries
        
        # Calculate ranks
        canary_ranks = self.calculate_ranks(format_string, randomness_space, all_canaries)
        
        # Calculate size of randomness space
        num_placeholders = format_string.count("_")
        space_size = len(randomness_space) ** num_placeholders
        
        # Calculate exposure for target canaries
        exposures = {}
        for canary in target_canaries:
            rank = canary_ranks[canary]
            exposure = np.log2(space_size) - np.log2(rank)
            exposures[canary] = max(0, exposure)  # Ensure exposure is non-negative
            
        return exposures

# Example usage
def example_usage():
    # Initialize calculator with a model
    calculator = CanaryExposureCalculator("gpt2")
    
    # Define format string and randomness space
    format_string = "The random number is _"
    randomness_space = list(range(10))  # Digits 0-9
    
    # Generate some target canaries
    target_canaries = ["The random number is 7", "The random number is 3"]
    
    # Calculate exposure
    exposures = calculator.calculate_exposure(format_string, randomness_space, target_canaries)
    
    # Print results
    for canary, exposure in exposures.items():
        print(f"Canary: {canary}")
        print(f"Exposure: {exposure:.2f} bits")
        print(f"Interpretation: {'High exposure (likely memorized)' if exposure > 3 else 'Low exposure'}")
        print()

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    example_usage()